# 01 · Distillation basics

Train two identical students on identical data: one on **hard labels**, one on the
**teacher's soft distribution**. Compare label efficiency and agreement with the teacher.

Runs on CPU in about a minute, no downloads: the "teacher" is a hand-written scoring
function over synthetic text features, so the soft targets are non-trivial but reproducible.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
N_CLASSES, N_FEATURES = 6, 60
CLASS_NAMES = [f"intent_{i}" for i in range(N_CLASSES)]


def make_dataset(n):
    """Synthetic text features with a class-dependent signal, plus label noise."""
    y = rng.integers(0, N_CLASSES, n)
    X = rng.normal(0, 1.0, (n, N_FEATURES))
    # each class shifts a distinct block of features
    for c in range(N_CLASSES):
        lo = (c * N_FEATURES) // N_CLASSES
        hi = ((c + 1) * N_FEATURES) // N_CLASSES
        X[y == c, lo:hi] += 0.6
    # 8% label noise: the ceiling every student inherits
    flip = rng.random(n) < 0.12
    y[flip] = rng.integers(0, N_CLASSES, flip.sum())
    return X, y


def teacher_probs(X, temperature=2.0):
    """A stand-in teacher: a *non-linear* function of the class-specific features.

    Non-linearity matters: a linear teacher would be recovered exactly by a linear
    student, and every comparison below would be trivially perfect.
    """
    z = np.zeros((len(X), N_CLASSES))
    for c in range(N_CLASSES):
        lo = (c * N_FEATURES) // N_CLASSES
        hi = ((c + 1) * N_FEATURES) // N_CLASSES
        block = X[:, lo:hi]
        z[:, c] = 2.5 * np.tanh(1.4 * block.mean(axis=1)) - 0.8 * block.std(axis=1) ** 2
    z = z / temperature
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


X_train, y_train = make_dataset(8000)
X_test, y_test = make_dataset(2000)
T_train = teacher_probs(X_train)
T_test = teacher_probs(X_test)
print(X_train.shape, T_train.shape, "teacher accuracy:",
      (T_test.argmax(1) == y_test).mean().round(3))

## The student

A multinomial logistic regression trained with plain gradient descent — small enough to
read, and enough to show the effect. The only difference between the two runs is the loss.


In [ ]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def train_student(X, targets, epochs=150, lr=0.5, l2=1e-4, soft=True, T=4.0):
    """targets: hard labels (soft=False) or a probability matrix (soft=True)."""
    n, d = X.shape
    W = np.zeros((d, N_CLASSES))
    b = np.zeros(N_CLASSES)
    for _ in range(epochs):
        p = softmax(X @ W + b)
        if soft:
            # KL against the temperature-softened teacher, scaled by T^2
            t = targets ** (1.0 / T)
            t = t / t.sum(axis=1, keepdims=True)
            q = softmax((X @ W + b) / T)
            grad_logits = (q - t) * T / n          # T^2 * d/dz KL, T cancels once
        else:
            onehot = np.eye(N_CLASSES)[targets]
            grad_logits = (p - onehot) / n
        W -= lr * (X.T @ grad_logits + l2 * W)
        b -= lr * grad_logits.sum(axis=0)
    return W, b


def evaluate(W, b, X, y):
    return float((softmax(X @ W + b).argmax(1) == y).mean())


def agreement(W, b, X, teacher_p):
    return float((softmax(X @ W + b).argmax(1) == teacher_p.argmax(1)).mean())

In [ ]:
W_hard, b_hard = train_student(X_train, y_train, soft=False)
W_soft, b_soft = train_student(X_train, T_train, soft=True)

print("hard-label student : acc %.3f  agreement %.3f" %
      (evaluate(W_hard, b_hard, X_test, y_test), agreement(W_hard, b_hard, X_test, T_test)))
print("soft-target student: acc %.3f  agreement %.3f" %
      (evaluate(W_soft, b_soft, X_test, y_test), agreement(W_soft, b_soft, X_test, T_test)))

## Label efficiency

The interesting question is not who wins with 8,000 examples — it is who wins with 100.


In [ ]:
rows = []
for n in [100, 250, 500, 1000, 2000, 4000, 8000]:
    Xs, ys, Ts = X_train[:n], y_train[:n], T_train[:n]
    Wh, bh = train_student(Xs, ys, soft=False, epochs=200)
    Ws, bs = train_student(Xs, Ts, soft=True, epochs=200)
    rows.append({"n_train": n,
                 "hard_acc": round(evaluate(Wh, bh, X_test, y_test), 3),
                 "soft_acc": round(evaluate(Ws, bs, X_test, y_test), 3)})

df = pd.DataFrame(rows)
df["gap"] = (df.soft_acc - df.hard_acc).round(3)
print(df.to_string(index=False))

## Temperature sweep — measure, do not assume

`T = 4` is folklore, not a law. Here the argmax is insensitive to T because the student
has enough capacity to fit the teacher's distribution either way; the sweep is included so
you can see that, rather than take a canonical value on faith. Temperature earns its keep
when the student is capacity-limited, data-starved, or when you need the *probabilities*
(calibration, cost thresholds) rather than just the argmax.


In [ ]:
# at 8,000 examples temperature barely matters; at 500 it is the main knob
for n in [500, 8000]:
    print("n =", n)
    for T in [1, 2, 4, 8]:
        W, b = train_student(X_train[:n], T_train[:n], soft=True, T=T, epochs=200)
        print("   T=%d  acc %.3f  agreement %.3f" %
              (T, evaluate(W, b, X_test, y_test), agreement(W, b, X_test, T_test)))

## Takeaways

- The soft-target advantage is largest when data is scarce, and shrinks as n grows.
- Agreement with the teacher plateaus below 1.0 — the student learns the teacher's
  *decision boundary*, not its exact distribution.
- Label noise (8% here) sets a ceiling that no distillation trick can lift. Fix labels
  before you tune temperature.
